In [14]:

import numpy as np
from scipy.stats import norm
from typing import Union, List

def black_scholes_eur_call(
    r: float, # risk-free interest rate
    T: float,  #time to maturity in years
    S0: float, #current stock price
    sigma: float, #stock volatility
    K: Union[float, List[float], np.ndarray] #strike price or an array of strike prices
) -> dict:

    # Check input conditions.
    assert sigma > 0, "Volatility must be positive."
    assert T > 0, "Time to maturity must be positive."
    assert S0 > 0, "Stock price must be positive."

    #Convert strike prices into a NumPy array.
    K_vec = np.atleast_1d(np.asarray(K, dtype=float))

    # Check that all strike prices are positive.
    assert np.all(K_vec > 0), "Strike prices must be positive."

    # Calculate d1 and d2
    d1_vec = (
        np.log(S0 / K_vec)
        + (r + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))

    d2_vec = d1_vec - sigma * np.sqrt(T)

    # Calculate standard normal cumulative probabilities.
    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    # Calculate the European call option price.
   
    call_price = (
        S0 * N_d1_vec
        - K_vec * np.exp(-r * T) * N_d2_vec
    )

    # Calculate the standard normal density at d1.
    pdf_d1_vec = norm.pdf(d1_vec)

    # Calculate Delta: sensitivity to the stock price.
    delta = N_d1_vec

    # Calculate Gamma
    gamma = (
        pdf_d1_vec / (S0 * sigma * np.sqrt(T))
    )

    # Calculate Vega: sensitivity to volatility.

    vega = S0 * pdf_d1_vec * np.sqrt(T)

    # Calculate Theta: sensitivity to calendar time.
    
    theta = (
        -(S0 * sigma * pdf_d1_vec) / (2 * np.sqrt(T))
        - r * K_vec * np.exp(-r * T) * N_d2_vec
    )

    #Calculate Rho: sensitivity to the risk-free rate.
    rho = K_vec * T * np.exp(-r * T) * N_d2_vec

    # Return the price and all Greeks in a dictionary.
    return {
        'price': call_price,
        'delta': delta,
        'gamma': gamma,
        'vega': vega,
        'theta': theta,
        'rho': rho
    }


In [15]:

r = 0.05       # Risk-free interest rate 
T = 1.0        # Time to maturity 
S0 = 100.0     # Current stock price 
sigma = 0.20   # Volatility 
K = 100.0      # Strike price 

# Call the modified Black-Scholes function.

results = black_scholes_eur_call(r, T, S0, sigma, K)

# Display the option price and all Greeks.

for name, value in results.items():
    print(f"{name.capitalize():8s}: {value[0]:.6f}")


Price   : 10.450584
Delta   : 0.636831
Gamma   : 0.018762
Vega    : 37.524035
Theta   : -6.414028
Rho     : 53.232482


In [16]:

# Calculate option prices and Greeks for different strike prices.
volatilities = [0.10, 0.20, 0.30, 0.40, 0.50]
r = 0.05
T = 1.0
S0 = 100.0
sigma = 0.20

K_values = [80, 90, 100, 110, 120]

# Call the modified BSM function.
results = black_scholes_eur_call(r, T, S0, sigma, K_values)

# Display all calculated results.
print(f"{'K':>5} {'Price':>10} {'Delta':>10} {'Gamma':>10} {'Vega':>10} {'Theta':>10} {'Rho':>10}")

for i, K in enumerate(K_values):
    print(
        f"{K:5.0f} "
        f"{results['price'][i]:10.4f} "
        f"{results['delta'][i]:10.4f} "
        f"{results['gamma'][i]:10.4f} "
        f"{results['vega'][i]:10.4f} "
        f"{results['theta'][i]:10.4f} "
        f"{results['rho'][i]:10.4f}"
    )


    K      Price      Delta      Gamma       Vega      Theta        Rho
   80    24.5888     0.9286     0.0068    13.6272    -4.7765    68.2749
   90    16.6994     0.8097     0.0136    27.1626    -5.9298    64.2709
  100    10.4506     0.6368     0.0188    37.5240    -6.4140    53.2325
  110     6.0401     0.4496     0.0198    39.5760    -5.9038    38.9247
  120     3.2475     0.2872     0.0170    34.0738    -4.6810    25.4717


In [17]:

r = 0.04
T = 2.0
S0 = 100.0
K = 100.0

volatilities = [0.10, 0.20, 0.30, 0.40, 0.50]

print(f"{'Volatility':>12} {'Price':>12} {'Vega':>12}")

for sigma in volatilities:
    results = black_scholes_eur_call(r, T, S0, sigma, K)

    print(
        f"{sigma:12.2f} "
        f"{results['price'][0]:12.4f} "
        f"{results['vega'][0]:12.4f}"
    )


  Volatility        Price         Vega
        0.10      10.1076      46.0766
        0.20      15.0847      51.5630
        0.30      20.2798      52.0668
        0.40      25.4663      51.5630
        0.50      30.5774      50.5977
